# Fase 2 — agente, ferramentas e ViaCEP

Esta demonstração importa a lógica dos arquivos `.py`. Catálogo, preços, aplicações e distâncias são fictícios. No modo `real`, OpenAI e ViaCEP são chamados de fato; no modo `simulado`, as decisões e o transporte são roteirizados e identificados.

Cada mensagem é independente. Memória, FAISS, text-to-SQL e avaliação integrada ficam para fases posteriores. A matriz de requisitos registra as equivalências ainda sujeitas à confirmação do enunciado original.

In [1]:
# AMBIENTE localiza somente o pacote do projeto, sem depender do repositório externo.
import os
import sys
import json
from pathlib import Path
from importlib.metadata import version

projeto = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "INSTRUCOES_PARA_IA_PROGRAMADORA.md").is_file())
sys.path.insert(0, str(projeto / "02_codigo/src"))
from auto_super_parts.agente_fase02 import AtendimentoFase02, INSTRUCOES_AGENTE
from auto_super_parts.demonstracao_fase02 import (executar_caso, esquemas_para_estudo, modelo_simulado, CASOS, demonstrar_falhas_cep)
from auto_super_parts.viacep import criar_cliente_dummy

modo = os.environ.get("FASE02_MODO", "real")
assert modo in ("real", "simulado")
print("Python:", sys.executable)
print("Modo:", modo, "| Projeto:", projeto.name)
print({n: version(n) for n in ("langchain", "langchain-core", "langchain-openai", "langgraph", "pydantic")})

Python: C:\Users\Renan\Documents\GIT_PROJETOS\estudos-engenharia-ia\.venv\Scripts\python.exe
Modo: simulado | Projeto: Projeto final
{'langchain': '1.4.7', 'langchain-core': '1.7.0', 'langchain-openai': '1.7.0', 'langgraph': '1.2.17', 'pydantic': '2.14.0'}


## Prompt e escolha de ferramentas

A persona, tarefa, contexto e formato definem o atendimento. O agente escolhe consultas; o cálculo usa preços do banco e regras em Python. O esquema enviado ao provedor usa modo estrito, mas BaseModel continua validando os argumentos localmente.

In [2]:
# SYSTEM_PROMPT define as regras; @TOOL expõe nome, descrição e argumentos tipados.
print(INSTRUCOES_AGENTE)
sistema_estudo = AtendimentoFase02(modelo=modelo_simulado(CASOS["retirada"]), cliente_cep=criar_cliente_dummy())
try:
    for ferramenta in sistema_estudo.ferramentas:
        print(ferramenta.name, ":", ferramenta.description)
    esquemas = esquemas_para_estudo(sistema_estudo)
finally:
    sistema_estudo.fechar()
# JSON SCHEMA: minItems é restrição de itens, nunca parâmetro da função.
print(json.dumps(esquemas, ensure_ascii=False, indent=2))

PERSONA: Você é atendente da Auto Super Parts CWB, loja fictícia, na fase 2.
TAREFA E CONTEXTO:
Sua tarefa é responder em português usando as quatro ferramentas disponíveis. Catálogo, aplicações,
marcas, preços, estoque e distâncias são simulação acadêmica, não homologação técnica nem cotação real.
Não coloque dados ou respostas do catálogo no prompt nem use memória de outra execução.
Para código de produto informado: consulte esse ID/SKU. Para peça por nome, use listar_veiculos
e peça os dados necessários para identificar a configuração; não escolha versão, 8V ou combustível
que o cliente não informou. Uma configuração listada não prova ser a do cliente. Com configuração
completa correspondente à lista, consulte aplicações por veiculo_id e categoria. Filtro e correia
genéricos exigem esclarecimento do tipo. Não invente IDs de produtos ou veículos.
Produto com estoque zero existe, mas está sem saldo. Resultado vazio não é preço zero.
Para orçamento, consulte os produtos e chame calcula

## Retirada e cálculo independente

Duas unidades de P003 custam 2 × 1850 = 3700 centavos. A demonstração mostra chamadas escolhidas, argumentos, retornos e resposta final. O relatório compara a trajetória com a expectativa, sem substituir o observado.

In [3]:
# INVOKE executa modelo -> ferramenta -> modelo; o helper imprime a trajetória e salva a evidência.
retirada = executar_caso("retirada", modo)

Modo: simulado | Caso: retirada | Mensagem: Quero 2 unidades do P003 para retirada. Consulte preço e estoque e calcule o orçamento.
[tool calls] [{"name": "consultar_catalogo", "args": {"produto_id": "P003"}, "id": "simulado_0", "type": "tool_call"}]
[retorno] consultar_catalogo: {"status": "ok", "produtos": [{"id": "P003", "sku": "CWB-SIM-P003", "nome": "Vela de ignição — Gol (simulação)", "categoria": "vela", "marca": "CWB Didática — fictícia", "codigo_fabricante_real": null, "unidade": "unidade", "preco_centavos": 1850, "estoque": 16, "atributos": {}, "natureza": "produto_ficticio_para_exercicio"}], "aplicacoes": [{"id": "A003", "produto_id": "P003", "veiculo_id": "V001", "status": "simulada_na_base_academica", "validada_para_uso_real": false, "fonte_aplicacao_real": null, "restricao": "Vínculo didático, não atestado de compatibilidade mecânica. Para óleo, especificações técnicas ainda pendentes."}], "sql_executado": [{"sql": "SELECT DISTINCT p.dados FROM produtos p WHERE (p.id = ? 

## Destino com distância cadastrada

O ViaCEP deve fornecer Ponta Grossa/PR para o CEP de teste; a base de distâncias fictícias contém 120 km. O frete esperado é 120 × 885 / 200 = 531 centavos; total de 4231 centavos e 4 dias úteis. O início da contagem permanece pendente, portanto não há promessa de data de calendário.

In [4]:
# API + PYTHON: o município/UF deve vir do retorno de consultar_cep antes do cálculo.
cadastrada = executar_caso("entrega_cadastrada", modo)

Modo: simulado | Caso: entrega_cadastrada | Mensagem: Consulte o CEP 84010000 no ViaCEP e calcule 2 unidades do P003 para entrega nesse endereço.
[tool calls] [{"name": "consultar_catalogo", "args": {"produto_id": "P003"}, "id": "simulado_0", "type": "tool_call"}]
[retorno] consultar_catalogo: {"status": "ok", "produtos": [{"id": "P003", "sku": "CWB-SIM-P003", "nome": "Vela de ignição — Gol (simulação)", "categoria": "vela", "marca": "CWB Didática — fictícia", "codigo_fabricante_real": null, "unidade": "unidade", "preco_centavos": 1850, "estoque": 16, "atributos": {}, "natureza": "produto_ficticio_para_exercicio"}], "aplicacoes": [{"id": "A003", "produto_id": "P003", "veiculo_id": "V001", "status": "simulada_na_base_academica", "validada_para_uso_real": false, "fonte_aplicacao_real": null, "restricao": "Vínculo didático, não atestado de compatibilidade mecânica. Para óleo, especificações técnicas ainda pendentes."}], "sql_executado": [{"sql": "SELECT DISTINCT p.dados FROM produtos p WH

## Destino sem distância cadastrada

São Paulo/SP não tem distância na base fornecida. O subtotal pode ser conhecido, mas frete, prazo e total com entrega devem permanecer desconhecidos. A tentativa anterior com argumento extra minItems continua preservada no relatório da revisão.

In [5]:
# AUSÊNCIA não equivale a zero: a ferramenta deve retornar pendência, sem orçamento de entrega inventado.
sem_distancia = executar_caso("entrega_sem_distancia", modo)

Modo: simulado | Caso: entrega_sem_distancia | Mensagem: Consulte o CEP 01001000 no ViaCEP e calcule 2 unidades do P003 para entrega nesse endereço. Se faltar distância cadastrada, informe a pendência.
[tool calls] [{"name": "consultar_catalogo", "args": {"produto_id": "P003"}, "id": "simulado_0", "type": "tool_call"}]
[retorno] consultar_catalogo: {"status": "ok", "produtos": [{"id": "P003", "sku": "CWB-SIM-P003", "nome": "Vela de ignição — Gol (simulação)", "categoria": "vela", "marca": "CWB Didática — fictícia", "codigo_fabricante_real": null, "unidade": "unidade", "preco_centavos": 1850, "estoque": 16, "atributos": {}, "natureza": "produto_ficticio_para_exercicio"}], "aplicacoes": [{"id": "A003", "produto_id": "P003", "veiculo_id": "V001", "status": "simulada_na_base_academica", "validada_para_uso_real": false, "fonte_aplicacao_real": null, "restricao": "Vínculo didático, não atestado de compatibilidade mecânica. Para óleo, especificações técnicas ainda pendentes."}], "sql_executad

## Tratamento de falhas reproduzíveis

Os próximos cenários de CEP inexistente e timeout são simulados explicitamente. Não afirmam que a API real falhou. Não há fallback silencioso para dados dummy.

In [6]:
# INJEÇÃO DE TRANSPORTE demonstra erros sem rede; CEP inválido é rejeitado localmente.
falhas = demonstrar_falhas_cep()
assert falhas["entrada_invalida"]["status"] == "cep_invalido"
assert falhas["nao_encontrado_simulado"]["status"] == "cep_nao_encontrado"
assert falhas["timeout_simulado"]["status"] == "timeout"

{
  "entrada_invalida": {
    "origem": "transporte_injetado_teste",
    "cep": "123",
    "endereco": null,
    "tentativa_http_real": false,
    "status": "cep_invalido",
    "pendencias": [
      "Informe CEP com oito dígitos."
    ]
  },
  "nao_encontrado_simulado": {
    "origem": "transporte_injetado_teste",
    "cep": "99999999",
    "endereco": null,
    "tentativa_http_real": false,
    "url": "https://viacep.com.br/ws/99999999/json/",
    "codigo_http": 200,
    "status": "cep_nao_encontrado"
  },
  "timeout_simulado": {
    "origem": "transporte_injetado_teste",
    "cep": "01001000",
    "endereco": null,
    "tentativa_http_real": false,
    "url": "https://viacep.com.br/ws/01001000/json/",
    "status": "timeout"
  }
}
[evidência] fase02_falhas_cep_simuladas_20261102T214234052327Z.json


In [7]:
# CONFERÊNCIA conserva cada observado antes de verificar; falhas impedem concluir o notebook como aprovado.
for relatorio in (retirada, cadastrada, sem_distancia):
    print(relatorio["caso"], relatorio["aprovado_dados_e_trajetoria"], relatorio["verificacoes"])
    assert relatorio["aprovado_dados_e_trajetoria"], relatorio["caso"]
if modo == "real":
    # CALLBACK mostra o esquema capturado na chamada real; nenhum segredo é registrado.
    enviados = sem_distancia["observado"]["esquemas_modelo"]
    assert enviados
    for ferramentas in enviados:
        assert all(t["function"]["strict"] for t in ferramentas)
    print("Esquema capturado na execução real:")
    print(json.dumps(enviados[0], ensure_ascii=False, indent=2))
print("Três cenários conferidos. Inspecione também o texto final e suas limitações.")

retirada True {'execucao_ok': True, 'sem_argumentos_invalidos': True, 'calculo_executado': True, 'calculo.status': True, 'calculo.subtotal_centavos': True, 'calculo.frete_centavos': True, 'calculo.total_centavos': True, 'catalogo_antes_calculo': True, 'subtotal_independente': True}
entrega_cadastrada True {'execucao_ok': True, 'sem_argumentos_invalidos': True, 'calculo_executado': True, 'calculo.status': True, 'calculo.subtotal_centavos': True, 'calculo.frete_centavos': True, 'calculo.total_centavos': True, 'calculo.prazo_dias_uteis': True, 'catalogo_antes_calculo': True, 'subtotal_independente': True, 'cep_antes_calculo': True, 'destino_do_retorno_cep': True}
entrega_sem_distancia True {'execucao_ok': True, 'sem_argumentos_invalidos': True, 'calculo_executado': True, 'calculo.status': True, 'calculo.subtotal_centavos': True, 'calculo.frete_centavos': True, 'calculo.total_centavos': True, 'calculo.prazo_dias_uteis': True, 'catalogo_antes_calculo': True, 'subtotal_independente': True, '

## Limites desta fase

A validação numérica e de trajetória não certifica toda resposta em linguagem natural. Não há reserva de produto, compra, homologação técnica de peças, memória entre mensagens ou garantia de disponibilidade comercial real.

Veja a matriz em `01_documentacao/MATRIZ_FASE_02.md` e as evidências em `04_avaliacao_e_evidencias/resultados/RESULTADOS_FASE_02.md`. Implementação com apoio de Codex/OpenAI. Referência: [function calling e modo estrito](https://developers.openai.com/api/docs/guides/function-calling).